# Subset-robustness: pruning configs vs γ

10 random 15-of-20 model subsets, 6 configs, γ sweep. Each plot is mean |S| over seeds (spread = std). The **exact optimum** line is exhaustive ground truth at large γ (the honest optimal — *not* the optimistic MILP, which we showed is loose by 2–3× at small γ).

## Setup — load results

In [ ]:
%matplotlib inline
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt

# works whether the notebook runs from notebooks/ or the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
RES = os.path.join(ROOT, "results", "subset_robustness")

payload = json.load(open(sorted(glob.glob(os.path.join(RES, "subset_robustness_*seeds.json")))[-1]))
summary = payload["summary"]
gammas = payload["config"]["gammas"]
n_seeds = payload["config"]["n_seeds"]
subset_size = payload["config"]["subset_size"]

CONFIG_ORDER = ["backward", "forward", "forward_backward",
                "backward_kswap", "forward_kswap", "forward_backward_kswap"]
configs = [c for c in CONFIG_ORDER if c in summary]
cmap = plt.get_cmap("tab10")
colors = {c: cmap(i) for i, c in enumerate(configs)}
x = np.array(gammas, float)

def mean_std(c):
    m = np.array([summary[c][str(g)]["mean_size"] for g in gammas])
    s = np.array([summary[c][str(g)]["std_size"] for g in gammas])
    return m, s

# exact optimum reference (honest, exhaustive at large gamma) if available
opt = None
_p = os.path.join(RES, "optimum.json")
if os.path.exists(_p):
    opt = json.load(open(_p))
def opt_xy():
    if not opt: return [], []
    items = sorted((int(g), v) for g, v in opt["mean"].items() if v is not None)
    return [g for g, _ in items], [v for _, v in items]

print("configs:", configs)
print("exact optimum:", opt["mean"] if opt else "(optimum.json not found yet)")

## Plot 1 — |S| vs γ (all configs, mean ± std)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for c in configs:
    m, s = mean_std(c)
    ax.plot(x, m, "-o", color=colors[c], lw=1.8, ms=4, label=c)
    ax.fill_between(x, m - s, m + s, color=colors[c], alpha=0.15)
ox, oy = opt_xy()
if ox:
    ax.plot(ox, oy, "k--o", lw=2, ms=4, label="exact optimum")
ax.set_xlabel("tolerance γ")
ax.set_ylabel("kept models |S|")
ax.set_title(f"|S| vs γ  ({n_seeds} random {subset_size}-model subsets, mean ± std)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## Plot 2 — models kept per config (grouped bars)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ng, nc = len(gammas), len(configs)
w = 0.8 / nc
base = np.arange(ng)
for i, c in enumerate(configs):
    m, s = mean_std(c)
    ax.bar(base + i * w, m, w, yerr=s, capsize=2, color=colors[c],
           label=c, error_kw={"elinewidth": 0.7})
ax.set_xticks(base + 0.4 - w / 2)
ax.set_xticklabels([str(int(g)) for g in gammas])
ax.set_xlabel("tolerance γ")
ax.set_ylabel("kept models |S|  (mean, error = std)")
ax.set_title(f"Models kept per config  ({n_seeds} random {subset_size}-model subsets)")
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.show()

## Plot 3 — best configs vs exact optimum (γ ≥ 80)

In [ ]:
best = ["backward", "backward_kswap", "forward_kswap", "forward_backward_kswap"]
mask = x >= 80
fig, ax = plt.subplots(figsize=(8, 5))
for c in best:
    m, s = mean_std(c)
    ax.plot(x[mask], m[mask], "-o", color=colors[c], lw=1.8, ms=5, label=c)
    ax.fill_between(x[mask], (m - s)[mask], (m + s)[mask], color=colors[c], alpha=0.15)
ox, oy = opt_xy()
if ox:
    ax.plot(ox, oy, "k--o", lw=2.2, ms=6, label="exact optimum")
ax.set_xlabel("tolerance γ")
ax.set_ylabel("kept models |S|")
ax.set_title("Best configs vs exact optimum  (γ ≥ 80)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## Plot 4 — gap to exact optimum (0 = provably optimal)

In [ ]:
ox, _ = opt_xy()
if not ox:
    print("optimum.json not found -- run the optimum computation first")
else:
    og = [g for g in gammas if int(g) in ox]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    nc = len(configs)
    w = 0.8 / nc
    base = np.arange(len(og))
    for i, c in enumerate(configs):
        gaps = [summary[c][str(g)]["mean_size"] - opt["mean"][str(int(g))] for g in og]
        ax.bar(base + i * w, gaps, w, color=colors[c], label=c)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(base + 0.4 - w / 2)
    ax.set_xticklabels([str(int(g)) for g in og])
    ax.set_xlabel("tolerance γ")
    ax.set_ylabel("mean |S| − exact optimum")
    ax.set_title("Gap to exact optimum  (0 = provably optimal)")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(axis="y", alpha=0.3)
    plt.show()